In [0]:
%run ./01_setup_environment

In [0]:
# ========================================
# 03_bronze_claims_ingestion
# ========================================
 
from pyspark.sql.functions import *
 
try:
 
    claims_df = spark.read.format("csv") \
        .option("header", True) \
        .option("inferSchema", True) \
        .load(f"{source_path}/claims")
 
    bronze_claims_df = claims_df.withColumn(
        "ingestion_time",
        current_timestamp()
    ).withColumn(
        "source_file",
        col("_metadata.file_path")
    )
 
    bronze_claims_df.write \
        .format("delta") \
        .mode("append") \
        .save(f"{bronze_path}/claims")
 
    log_audit(
        "claims_pipeline",
        "bronze",
        "bronze_claims",
        bronze_claims_df.count(),
        "SUCCESS"
    )
 
except Exception as e:
 
    log_audit(
        "claims_pipeline",
        "bronze",
        "bronze_claims",
        0,
        "FAILED",
        str(e)
    )
 
    raise e